# Push 2 Chord Drone

Chromatic keyboard + note toggle + chord bank on Push 2.

- **Rows 2-7**: Keyboard (live play) or Note Toggle (chord building), toggled via **Record** button
- **Row 0**: Chord bank — press to load, **Convert + press** to save
- **Row 1**: Free for live-coding experiments
- **Arrows / OctaveUp/Down**: Shift the grid
- **FBV 3 foot pedal**: CC 20-24 loads chord banks 0-4; channel 15 expression sets interpolation time
- limited to 3 simultaneous on notes in the MPE device

In [1]:
import { Push2 } from "@/push2/push2.ts"
import { MidiAccess } from "@/midi/mod.ts"
import { GridLayout } from "@/push2/grid_layout.ts"
import { KeyboardModule } from "@/push2/modules/keyboard.ts"
import { NoteToggleModule } from "@/push2/modules/note_toggle.ts"
import { BankModule } from "@/push2/modules/bank.ts"

const OUTPUT_NAME = 'IAC Driver Bus 1'
const FBV_NAME = 'FBV 3'

const midi = MidiAccess.open()
const outputs = midi.listOutputs()
console.log('MIDI outputs:', outputs.map(p => p.name))

const outputInfo = outputs.find(p => p.name.includes(OUTPUT_NAME))!
const iacOut = midi.openOutput(outputInfo.id)

const inputs = midi.listInputs()
const fbvInfo = inputs.find(p => p.name.includes(FBV_NAME))!
const fbvIn = midi.openInput(fbvInfo.id)

const push = Push2.create()
console.log('Push 2 connected, IAC output:', outputInfo.name, ', FBV 3 input:', fbvInfo.name)

MIDI outputs: [
  "IAC Driver Bus 1",
  "IAC Driver Bus 2",
  "IAC Driver Bus 3",
  "IAC Driver Bus 4",
  "IAC Driver Bus 5",
  "IAC Driver Bus 6",
  "IAC Driver Bus 7",
  "IAC Driver Bus 8",
  "Ableton Push 2 Live Port",
  "Ableton Push 2 User Port",
  "FBV 3",
  "Protokol"
]
Push 2 connected, IAC output: IAC Driver Bus 1 , FBV 3 input: FBV 3


In [2]:
import { COLOR } from "@/push2/constants.ts"
import { MPEDevice, type MPENoteRef } from "@/midi/mod.ts"

const CH = 0
const layout = new GridLayout({ rows: [2, 7], baseNote: 36, rowInterval: 5 })

// Drone notes tracked for keyboard guard
const droneNotes = new Set<number>()
const droneMpe = new MPEDevice(iacOut, {
  zone: "lower",
  memberChannels: [1, 3], // MIDI channels 2-4, one voice per chord note
  overflow: "none",
})
type DroneVoice = {
  ref: MPENoteRef
  baseNote: number
  visibleNote: number
  currentPitch: number
  bend: number
}
type DroneVisibleReassignment = { fromNote: number; toNote: number }

const droneVisibleToVoice = new Map<number, DroneVoice>()
const droneVoiceToVisible = new Map<number, number>()

function cancelFbvDroneIfActive(): void {
  const cancel = (globalThis as any)._fbvCancelInterpolation as (() => void) | undefined
  cancel?.()
}

function forgetDroneVoice(voice: DroneVoice): void {
  if (droneVisibleToVoice.get(voice.visibleNote) === voice) {
    droneVisibleToVoice.delete(voice.visibleNote)
  }
  droneVoiceToVisible.delete(voice.ref.id)
}

function droneNoteOn(note: number, velocity: number): void {
  const existing = droneVisibleToVoice.get(note)
  if (existing) {
    existing.ref.noteOff(0)
    forgetDroneVoice(existing)
  }
  const ref = droneMpe.noteOn(note, velocity, 0)
  if (ref) {
    const voice = { ref, baseNote: note, visibleNote: note, currentPitch: note, bend: 0 }
    droneVisibleToVoice.set(note, voice)
    droneVoiceToVisible.set(ref.id, note)
  } else {
    console.warn(`No MPE voice available for drone note ${note}`)
  }
}

function droneNoteOff(note: number): void {
  const voice = droneVisibleToVoice.get(note)
  if (!voice) return
  voice.ref.noteOff(0)
  forgetDroneVoice(voice)
}

function droneVoiceBaseNote(visibleNote: number): number | null {
  return droneVisibleToVoice.get(visibleNote)?.baseNote ?? null
}

function droneSetVoicePitch(visibleNote: number, pitch: number, bend: number): void {
  const voice = droneVisibleToVoice.get(visibleNote)
  if (!voice) return
  voice.currentPitch = pitch
  voice.bend = bend
  voice.ref.pitchBend(bend)
}

function reassignDroneVisibleNotes(reassignments: DroneVisibleReassignment[]): void {
  const moves: { fromNote: number; toNote: number; voice: DroneVoice }[] = []
  for (const { fromNote, toNote } of reassignments) {
    const voice = droneVisibleToVoice.get(fromNote)
    if (voice) moves.push({ fromNote, toNote, voice })
  }
  for (const { fromNote, voice } of moves) {
    if (droneVisibleToVoice.get(fromNote) === voice) droneVisibleToVoice.delete(fromNote)
  }
  for (const { toNote, voice } of moves) {
    voice.visibleNote = toNote
    droneVisibleToVoice.set(toNote, voice)
    droneVoiceToVisible.set(voice.ref.id, toNote)
  }
}

function resetDroneBends(): void {
  for (const voice of droneVisibleToVoice.values()) {
    voice.currentPitch = voice.baseNote
    voice.bend = 0
    voice.ref.pitchBend(0)
  }
}

const noteToggle = new NoteToggleModule(push, layout, {
  noteOn: (note, vel) => droneNoteOn(note, vel),
  noteOff: (note) => droneNoteOff(note),
})

// Sync drone state + keyboard highlights
noteToggle.onChange((notes) => {
  droneNotes.clear()
  for (const note of notes.keys()) droneNotes.add(note)
  const hl = new Map<number, number>()
  for (const note of notes.keys()) hl.set(note, COLOR.YELLOW)
  keyboard.setHighlights(hl)
})

const keyboard = new KeyboardModule(push, layout, {
  noteOn: (note, vel) => { if (!droneNotes.has(note)) iacOut.noteOn(CH, note, vel) },
  noteOff: (note) => { if (!droneNotes.has(note)) iacOut.noteOff(CH, note, 0) },
})

const bank = new BankModule<Map<number, number>>(
  push,
  (_index) => noteToggle.getOnNotes(),
  (chord) => {
    cancelFbvDroneIfActive()
    noteToggle.setNotes(chord)
    noteToggle.playAllOnNotes(true)
    resetDroneBends()
  },
  {
    rows: [0, 0],
    offFn: () => {
      cancelFbvDroneIfActive()
      noteToggle.allNotesOff()
    },
  },
)

// Start with keyboard active
keyboard.activate()
bank.activate()
let keyboardActive = true

push.onButtonPressed("Record", () => {
  if (keyboardActive) {
    keyboard.deactivate()
    noteToggle.activate()
  } else {
    noteToggle.deactivate()
    keyboard.activate()
  }
  keyboardActive = !keyboardActive
  console.log(keyboardActive ? 'Keyboard mode' : 'Note toggle mode')
})

// Re-push LED state when switching back from Ableton mode.
// The User button always sends on both ports, so releasing it
// after toggling back to User mode triggers a full LED refresh.
push.onButtonReleased("User", () => push.refreshLEDs())

console.log('Modules active. Record = toggle keyboard/note-toggle')

Modules active. Record = toggle keyboard/note-toggle


## Free Row (row 1)

Row 1 (i=1) is unassigned. Use this cell to experiment — e.g. map row 1 pads to trigger inversions, scene changes, etc. `push`, `iacOut`, `noteToggle`, `bank`, `layout` are all in scope.

In [ ]:
import { padIJToN, COLOR } from "@/push2/constants.ts"

// Store unsubs so re-running this cell replaces handlers instead of stacking
const _row1Unsubs: (() => void)[] = (globalThis as any)._row1Unsubs ?? []
_row1Unsubs.forEach(fn => fn())

const row1Unsubs: (() => void)[] = [];
(globalThis as any)._row1Unsubs = row1Unsubs

// Example: row 1 pads as 8 quick-trigger buttons
// Re-run this cell to redefine behavior on the fly
row1Unsubs.push(push.onPadPressed((_padN, [i, j], _vel) => {
  if (i !== 1) return
  console.log(`Row 1 pad [${i},${j}] pressed`)
  push.setPadColor(padIJToN(i, j), COLOR.RED)
}))
row1Unsubs.push(push.onPadReleased((_padN, [i, j]) => {
  if (i !== 1) return
  push.setPadColor(padIJToN(i, j), COLOR.BLACK)
}))

In [3]:
import { launch } from "@/copiedHelpers/offline_time_context.ts"

// FBV 3 foot pedal — re-run this cell to tweak behavior
// First, tear down any previous listeners / interpolation from re-runs
const _oldFbvDroneCleanup = (globalThis as any)._fbvDroneCleanup as (() => void) | undefined
_oldFbvDroneCleanup?.()

const fbvUnsubs: (() => void)[] = [];
(globalThis as any)._fbvUnsubs = fbvUnsubs

// CC 20-24, value 127 (button down) -> load chord banks 0-4
const FBV_BASE_CC = 20
const FBV_BANK_COUNT = 5
const FBV_EXPRESSION_CHANNEL = 15 // MIDI channel 15, one-based
const FBV_MAX_INTERP_SEC = 10
const FBV_INTERP_FPS = 60
const MPE_BEND_RANGE_SEMITONES = 48
const FBV_COMMIT_PROGRESS = 0.8

type FbvChordEntry = [note: number, velocity: number]
type FbvPair = { fromNote: number; toNote: number }

let fbvInterpSeconds = Number((globalThis as any)._fbvInterpSeconds ?? 3)
let fbvCurrentTask: ReturnType<typeof launch> | null = null

function chordEntries(chord: Map<number, number>): FbvChordEntry[] {
  return [...chord.entries()].sort(([a], [b]) => a - b)
}

function minimalMovementPairs(from: FbvChordEntry[], to: FbvChordEntry[]): FbvPair[] {
  const perms = [
    [0, 1, 2], [0, 2, 1], [1, 0, 2],
    [1, 2, 0], [2, 0, 1], [2, 1, 0],
  ]
  let bestPerm = perms[0]
  let bestCost = Infinity
  for (const perm of perms) {
    let cost = 0
    for (let i = 0; i < 3; i++) cost += Math.abs(to[perm[i]][0] - from[i][0])
    if (cost < bestCost) {
      bestCost = cost
      bestPerm = perm
    }
  }
  return from.map(([fromNote], i) => ({ fromNote, toNote: to[bestPerm[i]][0] }))
}

function bendForSemitones(delta: number): number {
  const bend = Math.round((delta / MPE_BEND_RANGE_SEMITONES) * 8192)
  return Math.max(-8192, Math.min(8191, bend))
}

function interpolatedPitch(pair: FbvPair, progress: number): number {
  return pair.fromNote + (pair.toNote - pair.fromNote) * progress
}

function bendForVoicePitch(visibleNote: number, pitch: number): number | null {
  const baseNote = droneVoiceBaseNote(visibleNote)
  if (baseNote === null) return null
  return bendForSemitones(pitch - baseNote)
}

function setFbvVoicePitch(visibleNote: number, pitch: number): void {
  const bend = bendForVoicePitch(visibleNote, pitch)
  if (bend === null) return
  droneSetVoicePitch(visibleNote, pitch, bend)
}

function snapFbvVoicesToVisibleNotes(): void {
  for (const note of noteToggle.getOnNotes().keys()) setFbvVoicePitch(note, note)
}

function applyFbvProgress(pairs: FbvPair[], progress: number, committed: boolean): void {
  const p = Math.max(0, Math.min(1, progress))
  for (const pair of pairs) {
    const pitch = interpolatedPitch(pair, p)
    const activeNote = committed ? pair.toNote : pair.fromNote
    setFbvVoicePitch(activeNote, pitch)
  }
}

function cancelFbvInterpolation(snapToVisible = true): void {
  const task = fbvCurrentTask
  fbvCurrentTask = null
  task?.cancel()
  if (snapToVisible) snapFbvVoicesToVisibleNotes()
}

function playFbvChordInstant(targetChord: Map<number, number>): void {
  cancelFbvInterpolation(false)
  noteToggle.setNotes(targetChord)
  noteToggle.playAllOnNotes(true)
  resetDroneBends()
}

function interpolateFbvChord(targetChord: Map<number, number>): void {
  const sourceChord = new Map(noteToggle.getOnNotes())
  const from = chordEntries(sourceChord)
  const to = chordEntries(targetChord)
  const duration = fbvInterpSeconds
  if (duration <= 0 || from.length !== 3 || to.length !== 3) {
    playFbvChordInstant(targetChord)
    return
  }

  cancelFbvInterpolation()
  const pairs = minimalMovementPairs(from, to)
  let committed = false

  const commitTarget = () => {
    if (committed) return
    reassignDroneVisibleNotes(pairs)
    noteToggle.rebaseNotes(targetChord)
    committed = true
  }

  let task!: ReturnType<typeof launch>
  task = launch(async (ctx) => {
    try {
      applyFbvProgress(pairs, 0, committed)
      while (ctx.progTime < duration) {
        await ctx.waitSec(1 / FBV_INTERP_FPS)
        const progress = Math.min(1, ctx.progTime / duration)
        if (!committed && progress >= FBV_COMMIT_PROGRESS) commitTarget()
        applyFbvProgress(pairs, progress, committed)
      }
      if (!committed) commitTarget()
      applyFbvProgress(pairs, 1, committed)
    } catch (err) {
      if (!ctx.isCanceled) console.error('FBV interpolation failed', err)
    } finally {
      if (fbvCurrentTask === task) fbvCurrentTask = null
    }
  }, { debugName: "fbv-chord-interpolation" })
  fbvCurrentTask = task
  task.handleCancel(() => { if (fbvCurrentTask === task) snapFbvVoicesToVisibleNotes() })
}

function loadFbvBank(idx: number): void {
  const item = bank.getSlot(idx)
  if (item === null) return
  interpolateFbvChord(new Map(item))
}

function updateFbvInterpolationTime(ctrlVal: number): void {
  const next = Math.max(0, Math.min(127, ctrlVal)) / 127 * FBV_MAX_INTERP_SEC
  if (Math.abs(next - fbvInterpSeconds) < 0.01) return
  fbvInterpSeconds = next
  ;(globalThis as any)._fbvInterpSeconds = fbvInterpSeconds
}

fbvUnsubs.push(fbvIn.onCC((evt) => {
  const idx = evt.ctrlNum - FBV_BASE_CC
  const isBankButton = idx >= 0 && idx < FBV_BANK_COUNT

  if (evt.channel === FBV_EXPRESSION_CHANNEL && !isBankButton) {
    updateFbvInterpolationTime(evt.ctrlVal)
  }

  if (isBankButton && evt.ctrlVal === 127) {
    loadFbvBank(idx)
  }
}));

(globalThis as any)._fbvCancelInterpolation = cancelFbvInterpolation;
(globalThis as any)._fbvDroneCleanup = () => {
  fbvUnsubs.forEach(fn => fn())
  cancelFbvInterpolation()
  delete (globalThis as any)._fbvCancelInterpolation
}

console.log(`FBV 3 listeners active — expression channel 15 sets interpolation ${fbvInterpSeconds.toFixed(2)}s / ${FBV_MAX_INTERP_SEC}s`)

FBV 3 listeners active — expression channel 15 sets interpolation 3.00s / 10s


Note toggle mode


No MPE voice available for drone note 56
No MPE voice available for drone note 56
No MPE voice available for drone note 56
No MPE voice available for drone note 56
No MPE voice available for drone note 56
No MPE voice available for drone note 56
No MPE voice available for drone note 56


In [ ]:
// Cleanup: silence notes, clear LEDs, close MIDI
noteToggle.allNotesOff()
keyboard.deactivate()
noteToggle.deactivate()
bank.deactivate()
// Tear down FBV listeners / interpolation state
const _fbvDroneCleanup = (globalThis as any)._fbvDroneCleanup as (() => void) | undefined
if (_fbvDroneCleanup) {
  _fbvDroneCleanup()
} else {
  const _fbvCleanup: (() => void)[] = (globalThis as any)._fbvUnsubs ?? []
  _fbvCleanup.forEach(fn => fn())
}
// Clear row 1
for (let j = 0; j < 8; j++) push.setPadColor(padIJToN(1, j), COLOR.BLACK)

push.close()
fbvIn.close()
iacOut.close()
midi.close()
console.log('Cleaned up')